### Loading libraries

In [ ]:
import pandas as pd
import numpy as np


### Reading raw CSV files and combining datasets

This code loads three separate honeybee datasets (apiary, hive, and inspection details) and merges them into one large, combined table. It then cleans this master table by converting dates, dropping some unneeded columns, and removing any rows that have missing values or are duplicates, resulting in a clean dataset ready for analysis.

In [2]:
# --- 0. Define the file path ---
# This is the folder where all your files are located.
base_path = '/Users/hrichaacharya/Desktop/hope/'

# --- 1. Load all the necessary CSV files ---
# Load the inspection, hive_info, apiary_info and weather data files
inspection_df = pd.read_csv(base_path + 'HCC_Inspections.csv')
hive_df = pd.read_csv(base_path + 'Hive_Information.csv')
apiary_df = pd.read_csv(base_path + 'Apiary_Information.csv')
weather_df = pd.read_csv(base_path + 'final_weather_features.csv')

In [3]:
hive_merged = hive_df.merge(apiary_df, on='ApiaryID', how='left')
inspect_merged = inspection_df.merge(hive_merged, on='HiveID', how='left')
inspect_merged['InsptDate'] = pd.to_datetime(inspect_merged['InsptDate'])
cp_inspect_merged = inspect_merged.copy(deep=True)
cp_inspect_merged = cp_inspect_merged.drop(columns=['InpsectionID','Percent_Met','Hive_Tag','Apiary'])
cp_inspect_merged.dropna(inplace=True)
cp_inspect_merged.drop_duplicates(inplace=True)
cp_inspect_merged.reset_index(drop=True, inplace=True)
cp_inspect_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   HiveID     2087 non-null   int64         
 1   InsptDate  2087 non-null   datetime64[ns]
 2   Brood      2087 non-null   float64       
 3   Bees       2087 non-null   float64       
 4   Queen      2087 non-null   float64       
 5   Food       2087 non-null   float64       
 6   Stressors  2087 non-null   float64       
 7   Space      2087 non-null   float64       
 8   Healthy    2087 non-null   object        
 9   ApiaryID   2087 non-null   int64         
 10  City       2087 non-null   object        
 11  State      2087 non-null   object        
dtypes: datetime64[ns](1), float64(6), int64(2), object(3)
memory usage: 195.8+ KB


#### Feature engineering

This code is performing time-series feature engineering on the bee inspection data. It converts the 'Healthy' target to a binary 1 or 0, then sorts the data by hive and date. Its main purpose is to create "lag" features by shifting the health metrics (like `Brood`, `Queen`, `Healthy`, etc.) from the previous inspection onto the current row. It also adds temporal features like `Days_Since_Last_Inspection` and `Hive_Age_Days`, then cleans up the DataFrame by dropping the *current* health metrics (to prevent data leakage) and reordering the columns before saving the final result to a new CSV file - target_markers_lagged.csv.

In [4]:
# Feature Engineering and Selection
final_df = cp_inspect_merged.copy(deep=True)

# Convert target variable 'Healthy' to binary
final_df['Healthy'] = final_df['Healthy'].map({'Yes': 1, 'No': 0})

# Convert 'InsptDate' to datetime
final_df['InsptDate'] = pd.to_datetime(final_df['InsptDate'])
final_df = final_df.dropna(subset=['InsptDate'])

# Helper feature - Is first inspection
final_df = final_df.sort_values(by=['HiveID', 'InsptDate'])
final_df['Is_First_Inspection'] = final_df.groupby('HiveID').cumcount()
final_df['Is_First_Inspection'] = final_df['Is_First_Inspection'].apply(lambda x: 1 if x == 0 else 0)

# Creating lag features
# Define prev_brood_status based on previous inspection if is_first_inspection is 0 or NaN otherwise
final_df['Prev_Brood_Status'] = final_df.groupby('HiveID')['Brood'].shift(1)

# Define prev_bees_status based on previous inspection 
final_df['Prev_Bees_Status'] = final_df.groupby('HiveID')['Bees'].shift(1)

# Define prev_queen_status based on previous inspection 
final_df['Prev_Queen_Status'] = final_df.groupby('HiveID')['Queen'].shift(1)

# Define prev_food_status based on previous inspection
final_df['Prev_Food_Status'] = final_df.groupby('HiveID')['Food'].shift(1)

# Define prev_stressor_status based on previous inspection
final_df['Prev_Stressors_Status'] = final_df.groupby('HiveID')['Stressors'].shift(1)

# Define prev_space_status based on previous inspection
final_df['Prev_Space_Status'] = final_df.groupby('HiveID')['Space'].shift(1)

# Define prev_health_status based on previous inspection 
final_df['Prev_Health_Status'] = final_df.groupby('HiveID')['Healthy'].shift(1)

# Define number of days since last inspection
final_df = final_df.sort_values(by=['HiveID', 'InsptDate'])
final_df['Days_Since_Last_Inspection'] = final_df.groupby('HiveID')['InsptDate'].diff().dt.days

# Define Hive_Age_Days to store the age of the hive
first_inspection_date = final_df.groupby('HiveID')['InsptDate'].transform('min')    
final_df['Hive_Age_Days'] = (final_df['InsptDate'] - first_inspection_date).dt.days

current_health_cols_base = ['Brood', 'Bees', 'Queen', 'Food', 'Stressors', 'Space']
final_df.drop(columns=current_health_cols_base,inplace=True)

# Reordering the columns for convenient view
info_cols = ['State', 'City', 'ApiaryID', 'HiveID', 'InsptDate']
temporal_cols = ['Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days']

# Build the list of current and previous health features, paired together
prev_cols = []
for col in current_health_cols_base:
    prev_cols.append(f'Prev_{col}_Status')
prev_cols.append('Prev_Health_Status')
prev_cols.append('Healthy')

# Get all remaining columns
other_cols = [col for col in final_df.columns if col not in 
                (info_cols + temporal_cols + prev_cols)]

# Combine them in the desired order
new_column_order = (
    info_cols + 
    temporal_cols + 
    prev_cols + 
    other_cols 
)

final_df = final_df[new_column_order]
final_df.info()
final_df.to_csv(f"{base_path}target_markers_lagged.csv", index=False) 

<class 'pandas.core.frame.DataFrame'>
Index: 2087 entries, 0 to 2086
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   State                       2087 non-null   object        
 1   City                        2087 non-null   object        
 2   ApiaryID                    2087 non-null   int64         
 3   HiveID                      2087 non-null   int64         
 4   InsptDate                   2087 non-null   datetime64[ns]
 5   Is_First_Inspection         2087 non-null   int64         
 6   Days_Since_Last_Inspection  1900 non-null   float64       
 7   Hive_Age_Days               2087 non-null   int64         
 8   Prev_Brood_Status           1900 non-null   float64       
 9   Prev_Bees_Status            1900 non-null   float64       
 10  Prev_Queen_Status           1900 non-null   float64       
 11  Prev_Food_Status            1900 non-null   float64       
 1

#### Final data

This code merges the two datasets created in the previous steps: the `target_markers_lagged` file and the `final_weather_features` file. It loads both, then carefully prepares them for merging by converting the `InsptDate` columns in both tables to a consistent datetime format. To prevent errors, it de-duplicates the weather data, ensuring there is only one weather entry per inspection. Finally, it performs a left merge, adding the 7-day trailing weather features to the main bee inspection table using `HiveID` and `InsptDate` as the keys, and saves this final combined dataset to a new CSV file - final_merged_dataset.csv.

In [ ]:
weather_df['InsptDate'] = pd.to_datetime(weather_df['InsptDate'], errors='coerce')
weather_df.dropna(subset=['InsptDate'], inplace=True)
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   HiveID          2087 non-null   int64         
 1   InsptDate       2087 non-null   datetime64[ns]
 2   Healthy         2087 non-null   object        
 3   ApiaryID        2087 non-null   int64         
 4   City            2087 non-null   object        
 5   State           2087 non-null   object        
 6   Avg_prcp        2087 non-null   float64       
 7   Avg_wind        2087 non-null   float64       
 8   Avg_tmax        2087 non-null   float64       
 9   Avg_tmin        2087 non-null   float64       
 10  Avg_tavg        2087 non-null   float64       
 11  Avg_snow        2087 non-null   float64       
 12  Num_frost_days  2087 non-null   float64       
dtypes: datetime64[ns](1), float64(7), int64(2), object(3)
memory usage: 212.1+ KB


In [ ]:
try:
    # --- 1. Load Both Datasets ---
    print("Loading 'target_markers_lagged.csv' (our 'base' file)...")
    target_df = pd.read_csv(base_path + 'target_markers_lagged.csv')

    print(f"\nBase table ('target_df') shape: {target_df.shape}")
    print(f"Feature table ('weather_df') shape: {weather_df.shape}")

    # --- 2. Define Keys and Features for Merging ---
    merge_keys = ['HiveID', 'InsptDate']
    weather_features_to_add = [
        'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 
        'Avg_tavg', 'Avg_snow', 'Num_frost_days'
    ]
    
    # Create a clean version of the weather data with only keys + new features
    weather_df_clean = weather_df[merge_keys + weather_features_to_add]

    # --- 3. Clean the Merge Keys (CRITICAL STEP) ---
    print("Cleaning merge keys ('InsptDate' to datetime)...")
    target_df['InsptDate'] = pd.to_datetime(target_df['InsptDate'], errors='coerce')
    weather_df_clean['InsptDate'] = pd.to_datetime(weather_df_clean['InsptDate'], errors='coerce')

    # Drop any rows in *either* table that have an invalid date
    target_df = target_df.dropna(subset=['InsptDate'])
    weather_df_clean = weather_df_clean.dropna(subset=['InsptDate'])
    
    # --- 4. THE FIX: De-duplicate the Weather Data ---
    rows_before_drop = weather_df_clean.shape[0]
    
    # Drop any duplicate rows based on the merge keys
    weather_df_clean = weather_df_clean.drop_duplicates(subset=merge_keys, keep='first')
    
    rows_after_drop = weather_df_clean.shape[0]
    print(f"\nDe-duplicating weather data: {rows_before_drop} rows -> {rows_after_drop} rows")

    # --- 5. Perform the Merge ---
    print("Performing a LEFT merge...")
    
    final_df = pd.merge(
        target_df,
        weather_df_clean,
        on=merge_keys,
        how='left' # Keep all rows from target_df
    )

    # --- 6. Inspect the Result ---
    print("\n--- Merge Successful ---")
    print(f"Final merged DataFrame shape: {final_df.shape}")
    
    print("\n--- final_df info() ---")
    final_df.info()
    final_df.to_csv(f"{base_path}final_merged_dataset.csv", index=False)
    
    print("\n--- Checking for merge failures (NaNs in weather data) ---")
    nan_weather_count = final_df['Avg_tmax'].isna().sum()
    print(f"Rows with missing weather data after merge: {nan_weather_count}")

except Exception as e:
    print(f"An error occurred: {e}")

Loading 'target_markers_lagged.csv' (our 'base' file)...

Base table ('target_df') shape: (2087, 16)
Feature table ('weather_df') shape: (2087, 13)
Cleaning merge keys ('InsptDate' to datetime)...

De-duplicating weather data: 2087 rows -> 2081 rows
Performing a LEFT merge...

--- Merge Successful ---
Final merged DataFrame shape: (2087, 23)

--- final_df info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2087 entries, 0 to 2086
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   State                       2087 non-null   object        
 1   City                        2087 non-null   object        
 2   ApiaryID                    2087 non-null   int64         
 3   HiveID                      2087 non-null   int64         
 4   InsptDate                   2087 non-null   datetime64[ns]
 5   Is_First_Inspection         2087 non-null   int64         
 6   Days_Si

/var/folders/ln/ph4984y140jfjr4nlpdnmfcc0000gn/T/ipykernel_64384/494376415.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weather_df_clean['InsptDate'] = pd.to_datetime(weather_df_clean['InsptDate'], errors='coerce')
